<a href="https://colab.research.google.com/github/Shrxmy/25263CSDMachineLearning/blob/main/%5B6%5D%20Group%208%20-%20Module%206%20ANN%20Lab%20Number%202/Lab_4_Neural_Networks_Modified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANN Part 2: Neural Networks in NumPy and PyTorch

In this lab, you will build dense neural networks on the MNIST dataset **twice**:

1. **Using NumPy only** so that you can demonstrate your understanding of the forward pass, parameter representation, and core training computations.
2. **Using a deep learning framework (PyTorch)** so that you can compare your manual implementation with a standard library-based workflow.

For every major exercise, complete the **NumPy version first**, then complete the **PyTorch version**. At the end of the notebook, you must prepare a **summary table** comparing your implementations, settings, and results.


## Load the data and create train-test splits

In [ ]:
# Auto-setup when running on Google Colab
if 'google.colab' in str(get_ipython()):
    !pip install openml

!pip install pytorch_lightning
# General imports
%matplotlib inline
import numpy as np
import pandas as pd
import openml as oml
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

In [ ]:
# Download MNIST data. Takes a while the first time.
mnist = oml.datasets.get_dataset(554)
X, y, _, _ = mnist.get_data(target=mnist.default_target_attribute, dataset_format='array');
X = X.reshape(70000, 28, 28)

# Take some random examples
from random import randint
fig, axes = plt.subplots(1, 5,  figsize=(10, 5))
for i in range(5):
    n = randint(0,70000)
    axes[i].imshow(X[n], cmap=plt.cm.gray_r)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    axes[i].set_xlabel("{}".format(y[n]))
plt.show();

In [ ]:
# For MNIST, there exists a predefined stratified train-test split of 60000-10000. We therefore don't shuffle or stratify here.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=60000, random_state=0)

## Exercise 1: Preprocessing in NumPy, then in PyTorch

Complete the preprocessing pipeline in **two stages**:

### Part A. NumPy
* Normalize the data: map each feature value from its current representation (an integer between 0 and 255) to a floating-point value between 0 and 1.0.
* Use the given train-test split of 60,000 training examples and 10,000 testing examples.
* Flatten each image from shape `(28, 28)` to a vector of shape `(784,)`.
* Ensure your NumPy arrays have appropriate dtypes for the features and targets.

### Part B. PyTorch
* Starting from your processed NumPy arrays, convert the data to PyTorch tensors.
* Create a `TensorDataset` for the training data and another one for the testing data.

Your goal in this exercise is to show that you can preprocess the same dataset both manually with NumPy and in a format that can be consumed by a neural network framework.


In [ ]:
# Exercise 1A - NumPy preprocessing

X_train_np = X_train.reshape(X_train.shape[0], -1).astype(np.float32) / 255.0
X_test_np  = X_test.reshape(X_test.shape[0],  -1).astype(np.float32) / 255.0

y_train_np = y_train.astype(np.int64)
y_test_np  = y_test.astype(np.int64)

print(f'X_train_np shape: {X_train_np.shape}, dtype: {X_train_np.dtype}')
print(f'X_test_np  shape: {X_test_np.shape},  dtype: {X_test_np.dtype}')
print(f'y_train_np shape: {y_train_np.shape}, dtype: {y_train_np.dtype}')
print(f'y_test_np  shape: {y_test_np.shape},  dtype: {y_test_np.dtype}')
print(f'Feature range: [{X_train_np.min():.1f}, {X_train_np.max():.1f}]')

In [ ]:
# Exercise 1B - Convert the processed NumPy arrays to PyTorch tensors and datasets

X_train_tensor = torch.tensor(X_train_np)
X_test_tensor  = torch.tensor(X_test_np)
y_train_tensor = torch.tensor(y_train_np)
y_test_tensor  = torch.tensor(y_test_np)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

print(f'X_train_tensor: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}')
print(f'X_test_tensor:  {X_test_tensor.shape},  dtype: {X_test_tensor.dtype}')
print(f'y_train_tensor: {y_train_tensor.shape}, dtype: {y_train_tensor.dtype}')
print(f'y_test_tensor:  {y_test_tensor.shape},  dtype: {y_test_tensor.dtype}')
print(f'train_dataset length: {len(train_dataset)}')
print(f'test_dataset  length: {len(test_dataset)}')


## Exercise 2: Create the model in NumPy, then in PyTorch

In this exercise, you will define the neural network architecture in **two forms**.

### Part A. NumPy
Implement the building blocks of a two-layer neural network using NumPy. At minimum, define:
* a parameter initialization function
* a forward-pass function
* the activation functions you need

Represent the parameters explicitly (for example, weight matrices and bias vectors).

### Part B. PyTorch
Implement a `create_model` function which defines the topology of the deep neural network, specifying the following:
* The number of layers in the neural network: use **2 dense layers** for now (one hidden layer and one output layer).
* The number of nodes in each layer: these should be parameters of your function.
* Any regularization layers. Add at least one dropout layer.

Consider:
* What should be the shape of the input layer?
* Which output activation or output interpretation is appropriate for a **10-class classification** problem?
* How will your NumPy implementation and PyTorch implementation represent the same architecture?


In [ ]:
### Exercise 2A - NumPy model definition
def init_numpy_model(input_units=784, hidden_units=32, output_units=10, seed=42):
    """Initialize NumPy weights and biases for a two-layer neural network."""
    np.random.seed(seed)

    params = {
        "W1": np.random.randn(input_units, hidden_units) * 0.01,
        "b1": np.zeros((1, hidden_units)),
        "W2": np.random.randn(hidden_units, output_units) * 0.01,
        "b2": np.zeros((1, output_units))
    }

    return params
    pass

def relu_numpy(x):
    return np.maximum(0, x)

def softmax_numpy(x):
    x_shifted = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    pass

def forward_numpy(X, params):
    """Return the intermediate values and output probabilities/logits of the NumPy model."""
    W1, b1 = params["W1"], params["b1"]
    W2, b2 = params["W2"], params["b2"]

    # Layer 1
    Z1 = X @ W1 + b1
    A1 = relu_numpy(Z1)

    # Layer 2 (output)
    Z2 = A1 @ W2 + b2
    A2 = softmax_numpy(Z2)

    cache = {
        "Z1": Z1, "A1": A1,
        "Z2": Z2, "A2": A2
    }

    return A2, cache

params = init_numpy_model()
sample_X = X_train_np[:5]
sample_y = y_train_np[:5]

probs, cache = forward_numpy(sample_X, params)
preds = np.argmax(probs, axis=1)

fig, axes = plt.subplots(1, 5, figsize=(10,3))

for i in range(5):
    axes[i].imshow(sample_X[i].reshape(28,28), cmap='gray')
    axes[i].set_title(f"Pred: {preds[i]}\nTrue: {sample_y[i]}")
    axes[i].axis('off')

plt.show()

#Softmax output visualization
plt.figure()
plt.bar(range(10), probs[0])
plt.title("Softmax Output (Class Probabilities)")
plt.xlabel("Digit Class")
plt.ylabel("Probability")
plt.show()


# Hidden layer activation
plt.imshow(cache["A1"][0].reshape(1, -1), aspect='auto')
plt.title("Hidden Layer Activations (Sample 1)")
plt.colorbar()
plt.show()


#  Weight visualization
plt.imshow(params["W1"][:,0].reshape(28,28))
plt.title("Weights of First Hidden Neuron")
plt.colorbar()
plt.show()


### Exercise 2B - PyTorch model definition
def create_model(layer_1_units=32, layer_2_units=10, dropout_rate=0.3):
    model = nn.Sequential(
          nn.Linear(784, layer_1_units),
          nn.ReLU(),
          nn.Dropout(dropout_rate),
          nn.Linear(layer_1_units, layer_2_units)
    )
    return model
model = create_model()

sample_tensor = X_train_tensor[:5]
outputs = model(sample_tensor)
preds = torch.argmax(outputs, dim=1)

# Convert to numpy for plotting
sample_imgs = sample_tensor.numpy()

# Show predictions (PyTorch)
fig, axes = plt.subplots(1, 5, figsize=(10,3))

for i in range(5):
    axes[i].imshow(sample_imgs[i].reshape(28,28), cmap='gray')
    axes[i].set_title(f"Pred: {preds[i].item()}")
    axes[i].axis('off')

plt.show()

plt.figure()
plt.bar(range(10), outputs[0].detach().numpy())
plt.title("Model Output (Logits)")
plt.xlabel("Digit Class")
plt.ylabel("Score")
plt.show()

print(model)

## Exercise 3: Create the training procedure in NumPy, then in PyTorch

Implement the training logic for both approaches.

### Part A. NumPy
Create a training routine for your NumPy neural network. You may use full-batch gradient descent or mini-batch gradient descent, but you must clearly show:
* how predictions are computed
* how the loss is computed
* how gradients are obtained
* how the parameters are updated

Track the training loss and accuracy across epochs. If you implement validation tracking as well, that is even better.

### Part B. PyTorch
Implement a `train_model` function which trains and evaluates a given model. It should print or store the training and validation loss and accuracy for each epoch.


In [ ]:
def train_numpy_model(params, X_train, y_train, X_val=None, y_val=None, epochs=10, learning_rate=0.01, batch_size=None):
    """Train the NumPy neural network and return the trained parameters and history."""
    pass


def train_model(model, train_dataset, val_dataset, epochs=10, batch_size=64, learning_rate=0.001):
    """
    model: the model to train
    train_dataset: the training data and labels
    val_dataset: the validation data and labels
    epochs: the number of epochs to train for
    batch_size: the batch size for minibatch SGD
    learning_rate: the learning rate for the optimizer
    """
    pass


## Exercise 4: Evaluate both implementations

Evaluate your models under the following setup:
* learning rate = **0.003**
* epochs = **50**
* batch size = **4000**
* validation set = **20% of the total training data**

Use default settings otherwise. Then:

1. Train your **NumPy implementation** and report its final training/validation behavior and test-set performance.
2. Train your **PyTorch implementation** and report its final training/validation behavior and test-set performance.
3. Plot the learning curves of loss and accuracy.
4. Compare the two approaches in terms of correctness, training convenience, speed, readability, and performance.

Try to run the PyTorch model on GPU if one is available.

You may use the plotting function below, or improve it if needed.


In [ ]:
# Helper plotting function
#
# history: the history object returned by the training function

def plot_curve(history):
    """
    Plots the learning curves for accuracy and loss.

    history: Dictionary containing 'accuracy', 'val_accuracy', 'loss', 'val_loss' per epoch.
    """
    epochs = range(1, len(history["accuracy"]) + 1)

    plt.figure(figsize=(12, 5))

    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["accuracy"], label="Train Accuracy")
    plt.plot(epochs, history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curve")
    plt.legend()

    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.legend()

    plt.show()

## Exercise 5: Optimize and compare

Try to optimize your model, either manually or with a tuning method. At minimum, explore the following for the **PyTorch implementation**, and optionally for the **NumPy implementation** if time permits:
* the number of hidden layers
* the number of nodes in each layer
* the number of dropout layers and the dropout rate
* any other settings you think are important (for example, learning rate or optimizer choice)

Try to reach at least **96% test accuracy**.

After optimization, briefly explain which changes improved the model and which changes did not help.


## Exercise 6: Create a summary table

At the end of the notebook, prepare a **summary table** that compares your major experiments. Include at least the following rows:
* NumPy baseline model
* PyTorch baseline model
* Best optimized PyTorch model

Your table should include at least these columns:
* Implementation (`NumPy` or `PyTorch`)
* Model configuration
* Epochs
* Batch size
* Learning rate
* Final training accuracy
* Final validation accuracy
* Test accuracy
* Final training loss
* Final validation loss
* Key observations

After the table, write a short reflection discussing what you learned from implementing neural networks manually in NumPy versus using a framework.


In [ ]:
# Exercise 6 - Summary table template
summary_columns = [
    "Implementation",
    "Model configuration",
    "Epochs",
    "Batch size",
    "Learning rate",
    "Final training accuracy",
    "Final validation accuracy",
    "Test accuracy",
    "Final training loss",
    "Final validation loss",
    "Key observations",
]

summary_table = pd.DataFrame(columns=summary_columns)
summary_table
